In [1]:
import importlib
import numpy as np
import polars as pl
import scipy.sparse as sp
import torch
from tqdm import tqdm
from sklearn.decomposition import PCA
import umap

from datasets import DATA_FOLDER, Dataloader, prepare_interaction_data
from util import CHECKPOINT_FOLDER, get_checkpoint_filepath, load_checkpoint, load_config_from_checkpoint

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.mps.is_available() else torch.device("cpu")

DATASET = "ML-25M"
# SAE_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-3c29e9ee.ckpt"  # ELSA + Cosine
SAE_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-d6337b64.ckpt"  # ELSA + L2
# SAE_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-f634db83.ckpt"  # MultVAE + L2

sae_cfg = load_config_from_checkpoint(SAE_CHECKPOINT_PATH)
elsa_checkpoint_path = f"{CHECKPOINT_FOLDER}/{DATASET}/{sae_cfg['pretrained_model_checkpoint']}"
elsa_cfg = load_config_from_checkpoint(elsa_checkpoint_path)

interactions_df, train_csr, val_csr, test_csr, train_users, val_users, test_users, items = prepare_interaction_data(elsa_cfg)
train_users_to_idxs = {uid: uidx for uidx, uid in enumerate(train_users)}
val_users_to_idxs = {uid: uidx for uidx, uid in enumerate(val_users)}
test_users_to_idxs = {uid: uidx for uidx, uid in enumerate(test_users)}
items_to_idxs = {iid: iidx for iidx, iid in enumerate(items)}

items_df = (
    pl.scan_csv(f"{DATA_FOLDER}/{DATASET}/movies.csv").rename({"movieId": "item_id"}).cast({"item_id": pl.String}).cast({"item_id": pl.Categorical}).collect()
)

elsa_model_class = getattr(importlib.import_module(elsa_cfg["model_module"]), elsa_cfg["model_class"])
elsa = elsa_model_class(train_csr.shape[1], elsa_cfg["embedding_dim"]).to(device)
_, _ = load_checkpoint(elsa, None, get_checkpoint_filepath(elsa_cfg), device, None)

sae_model_class = getattr(importlib.import_module(sae_cfg["model_module"]), sae_cfg["model_class"])
sae_extra_params = {k: sae_cfg[k] for k in sae_cfg.keys() if k in ["l1_coef", "k"]}
sae = sae_model_class(elsa_cfg["embedding_dim"], sae_cfg["embedding_dim"], sae_cfg["reconstruction_loss"], l1_coef=sae_cfg["l1_coef"], k=sae_cfg["k"]).to(
    device
)
_, _ = load_checkpoint(sae, None, get_checkpoint_filepath(sae_cfg), device, None)

Removing users with < 5 interactions...
Dataset info: users=160776, items=40857, interactions=12448242
Train split info: users=128621, items=40857, interactions=9965146
Val split info: users=16078, items=40857, interactions=1234555
Test split info: users=16077, items=40857, interactions=1248541
Loaded checkpoint from checkpoints/ML-25M/ELSA-1024-10977915.ckpt (after 13 epochs)
Loaded checkpoint from checkpoints/ML-25M/TopKSAE-8192-d6337b64.ckpt (after 761 epochs)


In [2]:
elsa_embeddings = []
test_user_dataloader = Dataloader(test_csr, batch_size=1024, device=device)
with torch.no_grad():
    for onehot_batch in tqdm(test_user_dataloader):
        elsa_embeddings.append(elsa.encode(onehot_batch).cpu().numpy())
elsa_embeddings = np.vstack(elsa_embeddings)

elsa_embeddings.shape

100%|██████████| 16/16 [00:01<00:00, 11.93it/s]


(16077, 1024)

In [3]:
sparse_embeddings = []
onehot_items_dataloader = Dataloader(sp.eye(len(items), dtype=np.float32, format="csr"), batch_size=1024, device=device)
with torch.no_grad():
    for onehot_batch in tqdm(onehot_items_dataloader):
        elsa_embedding = elsa.encode(onehot_batch)  # Tensor, shape = (batch.shape[0] x elsa_cfg["embedding_dim"])
        sae_embedding, _, input_mean, input_std = sae.encode(elsa_embedding)
        sparse_embeddings.append(sp.csr_matrix(sae_embedding.cpu().numpy()))
sparse_embeddings = sp.vstack(sparse_embeddings)

neuron_is_alive = np.asarray(sparse_embeddings.sum(axis=0)).flatten() != 0
living_neurons = np.where(neuron_is_alive)[0]
dead_neuron_count = sparse_embeddings.shape[1] - neuron_is_alive.sum()
print(f"{dead_neuron_count} dead neurons out of {sparse_embeddings.shape[1]} ({dead_neuron_count / sparse_embeddings.shape[1]:.2%})")
print(f"{neuron_is_alive.sum()} alive ones")

100%|██████████| 40/40 [00:06<00:00,  6.31it/s]

4267 dead neurons out of 8192 (52.09%)
3925 alive ones


In [4]:
tag_df = (
    pl.scan_csv(f"{DATA_FOLDER}/{DATASET}/tags.csv")
    .rename({"userId": "user_id", "movieId": "item_id"})
    .cast({"user_id": pl.String, "item_id": pl.String})
    .cast({"user_id": pl.Categorical, "item_id": pl.Categorical})
    .with_columns(pl.col("tag").str.to_lowercase().str.strip_chars().alias("tag"))  # convert to lowercase and strip whitespace
    .filter(pl.col("tag").count().over("tag") >= 100)  # keep only tags assigned at least 100 times
    .filter(pl.col("item_id").is_in(items))  # keep only items with interactions
    .with_columns(pl.col("item_id").replace_strict(items_to_idxs).alias("item_idx"))
    .cast({"tag": pl.Categorical})
    .collect()
)

tag_df

sys:1: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance


user_id,item_id,tag,timestamp,item_idx
cat,cat,cat,i64,i64
"""3""","""260""","""classic""",1439472355,43
"""3""","""260""","""sci-fi""",1439472256,43
"""4""","""1732""","""dark comedy""",1573943598,181
"""4""","""1732""","""great dialogue""",1573943604,181
"""4""","""7569""","""so bad it's good""",1573943455,5010
…,…,…,…,…
"""162462""","""260""","""space""",1427470029,43
"""162492""","""260""","""classic sci-fi""",1436468895,43
"""162492""","""260""","""epic""",1436468882,43


In [5]:
def compute_tfidf(X):
    """
    Compute TF-IDF for a term-document value matrix.
    Parameters:
    X: np.ndarray (num_terms, num_documents)
    Returns:
    tfidf_matrix: np.ndarray (num_terms, num_documents) - TF-IDF values
    """
    # Compute Term Frequency (TF) - Normalize by column sum
    tf = X / np.sum(X, axis=0, keepdims=True)
    tf[np.isnan(tf)] = 0  # Handle division by zero
    # Compute Document Frequency (DF) - Count nonzero occurrences of each term
    df = np.count_nonzero(X, axis=1)
    # Compute Inverse Document Frequency (IDF) - Log-scaled
    num_documents = X.shape[1]
    idf = np.log((num_documents + 1) / (df + 1)) + 1  # Smoothing
    # Compute TF-IDF
    tfidf_matrix = tf * idf[:, np.newaxis]
    return tfidf_matrix


tags = tag_df["tag"].unique(maintain_order=True).to_numpy()
tag_item_counts = sp.csr_matrix(
    (
        np.ones(len(tag_df), dtype=np.float32),
        (
            tag_df["tag"].to_physical().to_numpy(),
            tag_df["item_idx"].to_numpy(),
        ),
    ),
    shape=(tag_df["tag"].n_unique(), len(items)),
)
tag_to_idx = {t: i for i, t in enumerate(tags)}

pti = tag_item_counts.copy()
pti.data /= pti.data.sum()
tag_neuron_activity = pti @ sparse_embeddings  # tag x neuron (CSR matrix)

# neuron -> tag that elicits most distinctive response in this neuron
top_tag_per_neuron = tags[compute_tfidf(tag_neuron_activity.toarray().T).argmax(axis=1)]  # term = neuron, document = tag
top_tag_per_neuron[~neuron_is_alive] = None

# neuron -> tag that best characterizes the neuron's overall activity
characteristic_tag_per_neuron = tags[compute_tfidf(tag_neuron_activity.toarray()).argmax(axis=0)]  # term = tag, document = neuron
characteristic_tag_per_neuron[~neuron_is_alive] = None

# tag -> neuron that most representatively encodes this tag
characteristic_neuron_per_tag = compute_tfidf(tag_neuron_activity.toarray().T).argmax(axis=0)  # term = neuron, document = tag

# tag -> neuron whose firing is most unique to that tag
top_neuron_per_tag = compute_tfidf(tag_neuron_activity.toarray()).argmax(axis=1)  # term = neuron, document = tag

print(pl.Series("top_neuron", top_neuron_per_tag).value_counts(sort=True))
print(pl.Series("top_tag", top_tag_per_neuron[neuron_is_alive]).value_counts(sort=True))

/tmp/ipykernel_2737897/2117447984.py:10: RuntimeWarning: invalid value encountered in divide
  tf = X / np.sum(X, axis=0, keepdims=True)


shape: (834, 2)
┌────────────┬───────┐
│ top_neuron ┆ count │
│ ---        ┆ ---   │
│ i64        ┆ u32   │
╞════════════╪═══════╡
│ 2041       ┆ 7     │
│ 5468       ┆ 7     │
│ 464        ┆ 7     │
│ 6535       ┆ 7     │
│ 7135       ┆ 7     │
│ …          ┆ …     │
│ 7920       ┆ 1     │
│ 5093       ┆ 1     │
│ 1356       ┆ 1     │
│ 2786       ┆ 1     │
│ 157        ┆ 1     │
└────────────┴───────┘
shape: (985, 2)
┌──────────────────┬───────┐
│ top_tag          ┆ count │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ audrey hepburn   ┆ 19    │
│ aardman          ┆ 18    │
│ woody allen      ┆ 18    │
│ sergio leone     ┆ 17    │
│ monty python     ┆ 16    │
│ …                ┆ …     │
│ prostitute       ┆ 1     │
│ darren aronofsky ┆ 1     │
│ wheelchair       ┆ 1     │
│ kick-butt women  ┆ 1     │
│ government       ┆ 1     │
└──────────────────┴───────┘


In [6]:
import plotly.graph_objects as go
from matplotlib import cm
from matplotlib.colors import Normalize, to_hex
from sklearn.preprocessing import normalize
from plotly.subplots import make_subplots


def steer_towards_concept(user_embeddings: np.ndarray, concept_idx: int, concept_boost: float) -> np.ndarray:
    sae_embeddings, _, input_mean, input_std = sae.encode(torch.tensor(user_embeddings).to(device))
    s = sae_embeddings.sum(-1, keepdim=True)
    sae_embeddings *= (1 - concept_boost) / s
    sae_embeddings[:, concept_idx] += concept_boost
    sae_embeddings *= s
    return sae.decode(sae_embeddings, input_mean, input_std).detach().cpu().numpy()


def plot_umap_steering(
    user_embeddings,
    item_embeddings,
    concepts=["love story", "david lynch"],
    sample_size=100,
    intermediate_dim=16,
    metric="cosine",
    n_neighbors=5,
    min_dist=0.1,
    random_state=42,
):
    boosts = [0.2, 0.4, 0.6, 0.8]
    colormaps = [cm.get_cmap(name) for name in ["Reds", "Blues", "Greens"]]
    norm = Normalize(vmin=-0.1, vmax=1.0)
    np.random.seed(random_state)

    concept_item_embeddings = item_embeddings[
        tag_df.filter(pl.col("tag").is_in(concepts))["item_id"].value_counts(sort=True)[:100, "item_id"].replace_strict(items_to_idxs).to_numpy()
    ]
    # pca = PCA(n_components=intermediate_dim, random_state=random_state).fit(concept_item_embeddings)
    user_sample = normalize(user_embeddings[np.random.choice(user_embeddings.shape[0], size=sample_size, replace=False)], norm="l2")
    pca = PCA(n_components=intermediate_dim, random_state=random_state).fit(np.vstack([normalize(user_sample, norm="l2"), concept_item_embeddings]))
    user_reduced = pca.transform(user_sample)
    proj = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, metric=metric, random_state=random_state).fit(user_reduced)

    fig = make_subplots(rows=1, cols=len(concepts), subplot_titles=[f"'{concepts[i]}'" for i in range(len(concepts))])
    X = proj.transform(user_reduced)
    for i, c in enumerate(concepts):
        fig.add_trace(
            go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers", marker=dict(size=12, color=to_hex(cm.get_cmap("Greys")(0.2), keep_alpha=False), opacity=1.0)),
            row=1,
            col=i + 1,
        )

    for i, c in enumerate(concepts):
        for boost in boosts:
            # cidx = top_neuron_per_tag[tag_to_idx[c]]
            cidx = characteristic_neuron_per_tag[tag_to_idx[c]]
            X = proj.transform(pca.transform(normalize(steer_towards_concept(user_sample, cidx, boost))))
            fig.add_trace(
                go.Scatter(
                    x=X[:, 0],
                    y=X[:, 1],
                    mode="markers",
                    marker=dict(size=12, color=to_hex(colormaps[i](norm(boost)), keep_alpha=False), opacity=0.2),
                ),
                row=1,
                col=i + 1,
            )

    for i, c in enumerate(concepts):
        representative_items = items_df.join(tag_df.filter(pl.col("tag") == c)["item_id"].value_counts(sort=True)[:3], on="item_id").with_columns(
            pl.col("item_id").replace_strict(items_to_idxs).alias("item_idx")
        )
        titles = representative_items["title"].to_numpy()
        X = proj.transform(pca.transform(item_embeddings[representative_items["item_idx"].to_numpy()]))
        fig.add_trace(
            go.Scatter(
                x=X[:, 0],
                y=X[:, 1],
                mode="markers",
                marker=dict(symbol="x", size=18, color=to_hex(colormaps[i](norm(0.75)), keep_alpha=False), opacity=1.0, line=dict(width=1, color="white")),
                text=titles,
                hoverinfo="text",
            ),
            row=1,
            col=i + 1,
        )

    fig.update_layout(
        title=f"UMAP projection of {sample_size} users steered in the direction of ",
        plot_bgcolor="white",
        showlegend=False,
        margin=dict(l=10, r=10, t=70, b=10),
        font=dict(size=16),
    )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)

    # Save and open in browser
    html_path = "umap_cluster_plot.html"
    fig.write_html(html_path, auto_open=False)

    fig.write_image("umap_cluster_plot.svg")

    fig.show()

In [10]:
plot_umap_steering(
    user_embeddings=elsa_embeddings,
    item_embeddings=elsa.encoder.detach().cpu().numpy(),
    concepts=["love story", "david lynch", "children"],
    # concepts=tag_df.sample(3)["tag"].to_numpy(),
    sample_size=1000,
    intermediate_dim=32,
    n_neighbors=7,
    min_dist=0.1,
    metric="cosine",
    random_state=44,
)

/tmp/ipykernel_2737897/2707498145.py:29: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.

/tmp/ipykernel_2737897/2707498145.py:34: CategoricalRemappingWarning:

Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance

/home/mspisak/miniconda3/envs/sae/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/home/mspisak/miniconda3/envs/sae/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/home/mspisak/miniconda3/envs/sae/lib/python3.11/site-packages/sklearn/utils/deprecat

In [8]:
items_df.join(tag_df.filter(pl.col("tag") == "children")["item_id"].value_counts(sort=True)[:3], on="item_id")

/tmp/ipykernel_2737897/1689650434.py:1: CategoricalRemappingWarning:

Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance



item_id,title,genres,count
cat,str,str,u32
"""1""","""Toy Story (1995)""","""Adventure|Animation|Children|C…",27
"""5618""","""Spirited Away (Sen to Chihiro …","""Adventure|Animation|Fantasy""",17
"""6377""","""Finding Nemo (2003)""","""Adventure|Animation|Children|C…",16
